# C2. Robustez del Random Forest (corrección ii)

Corrección ii: robustez del Random Forest evaluado por paciente.

Usa los mismos 25 pliegues por paciente que la Tabla 4.1 (pliegues_por_paciente.csv).
Hace dos cosas:

1. Matriz de confusión por paciente (Sección 4.2.2). Se toma la repetición 0:
   en ella cada una de las 616 pacientes cae en el pliegue de prueba exactamente
   una vez. Se arma con las probabilidades y los umbrales que guardó A2
   (`resultados/pred_A2.csv` y `metricas_A2.csv`), así que **hay que correr A2 antes**.
   El umbral de cada pliegue es el de Youden calculado con el entrenamiento (punto iii).

2. Chequeo de regularización (Sección 3.5.2). El modelo de la Tabla 4.1 usa
   max_depth=None y min_samples_leaf=1, lo que permite árboles muy profundos.
   Se prueban 9 combinaciones de max_depth y min_samples_leaf:
     a) AUC-ROC por paciente de cada combinación en los 25 pliegues externos;
     b) validación cruzada anidada: dentro de las pacientes de entrenamiento de
        cada pliegue, una validación interna de 3 pliegues, también agrupada por
        paciente, elige la combinación; luego se evalúa en el pliegue externo.

Abrir este notebook dentro de la carpeta donde están los CSV, `comun.py` y `pliegues_por_paciente.csv`, y correr todas las celdas.
Tarda unos minutos en una laptop.
Salidas en resultados/:
  matriz_confusion_RF.csv, MatrixConfusion_RF_paciente.png,
  robustez_RF_por_config.csv, robustez_RF_anidada.csv, robustez_RF_resumen.txt

In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

from comun import (ETIQUETA, RESULTADOS, leer_datos, cargar_pliegues, iterar_pliegues,
                   promedio_por_paciente)

RESULTADOS.mkdir(exist_ok=True)

## datos (igual que A2)

In [ ]:
metadata = leer_datos('metadata')
taxonomy = leer_datos('taxonomy_relabd.species')
full_data = pd.merge(metadata[['specimen', 'participant_id', ETIQUETA]], taxonomy, on='specimen')
X = full_data.drop(columns=['specimen', 'participant_id', ETIQUETA]).values
y = full_data[ETIQUETA].astype(int).values
pacientes = full_data['participant_id'].values
print(X.shape, '| pacientes:', len(set(pacientes)))

## combinaciones a probar

In [ ]:
PROFUNDIDADES = [None, 5, 10]
HOJAS = [1, 5, 10]
CONFIGS = list(product(PROFUNDIDADES, HOJAS))


def nombre(cfg):
    return f'max_depth={cfg[0]}, min_samples_leaf={cfg[1]}'


def ajustar_rf(cfg, X_tr, y_tr):
    rf = RandomForestClassifier(n_estimators=100, max_depth=cfg[0], min_samples_leaf=cfg[1],
                                class_weight='balanced', random_state=42, n_jobs=-1)
    return rf.fit(X_tr, y_tr)


def auc_paciente(rf, X_te, y_te, pac_te):
    pp = promedio_por_paciente(pac_te, y_te, rf.predict_proba(X_te)[:, 1])
    return roc_auc_score(pp.y, pp.prob), pp

## entrenamiento en los 25 pliegues (tarda unos minutos)

In [ ]:
pliegues = cargar_pliegues()
filas_cfg, filas_anidada = [], []

for rep, fold, ids_tr, ids_te in iterar_pliegues(pliegues):
    tr, te = np.isin(pacientes, list(ids_tr)), np.isin(pacientes, list(ids_te))

    # a) cada configuración en el pliegue externo
    for cfg in CONFIGS:
        rf = ajustar_rf(cfg, X[tr], y[tr])
        auc, pp = auc_paciente(rf, X[te], y[te], pacientes[te])
        filas_cfg.append(dict(rep=rep, fold=fold, max_depth=str(cfg[0]),
                              min_samples_leaf=cfg[1], auc_roc=auc))

    # b) selección anidada: validación interna por paciente sobre el entrenamiento
    X_tr, y_tr, pac_tr = X[tr], y[tr], pacientes[tr]
    interna = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=100 + 10 * rep + fold)
    auc_interno = {cfg: [] for cfg in CONFIGS}
    for i_tr, i_va in interna.split(X_tr, y_tr, groups=pac_tr):
        for cfg in CONFIGS:
            rf = ajustar_rf(cfg, X_tr[i_tr], y_tr[i_tr])
            auc_interno[cfg].append(auc_paciente(rf, X_tr[i_va], y_tr[i_va], pac_tr[i_va])[0])
    mejor = max(CONFIGS, key=lambda c: np.mean(auc_interno[c]))
    rf = ajustar_rf(mejor, X_tr, y_tr)
    auc_ext, _ = auc_paciente(rf, X[te], y[te], pacientes[te])
    filas_anidada.append(dict(rep=rep, fold=fold, max_depth=str(mejor[0]),
                              min_samples_leaf=mejor[1], auc_roc=auc_ext))
    print(f'rep {rep} fold {fold}: elegida {nombre(mejor)} -> AUC externo {auc_ext:.3f}')

## 1) matriz de confusión

In [ ]:
# Probabilidades y umbrales de A2 (mismo modelo que la Tabla 4.1), repetición 0
pred = pd.read_csv(RESULTADOS / 'pred_A2.csv')
umbrales = pd.read_csv(RESULTADOS / 'metricas_A2.csv').set_index(['rep', 'fold']).umbral
r0 = pred[pred.rep == 0].copy()
r0['pred'] = (r0.prob >= r0.apply(lambda f: umbrales[(f.rep, f.fold)], axis=1)).astype(int)
assert r0.participant_id.nunique() == len(r0) == 616
cm = confusion_matrix(r0.y, r0.pred, labels=[0, 1])
pd.DataFrame(cm, index=['Real: término', 'Real: pretérmino'],
             columns=['Predicho: término', 'Predicho: pretérmino']).to_csv(RESULTADOS / 'matriz_confusion_RF.csv')
tn, fp, fn, tp = cm.ravel()
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Término', 'Pretérmino'], yticklabels=['Término', 'Pretérmino'])
plt.title('Matriz de confusión por paciente (Random Forest)')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig(RESULTADOS / 'MatrixConfusion_RF_paciente.png', dpi=200)
plt.show()

## 2) regularización

In [ ]:
por_cfg = pd.DataFrame(filas_cfg)
por_cfg.to_csv(RESULTADOS / 'robustez_RF_por_config.csv', index=False)
anidada = pd.DataFrame(filas_anidada)
anidada.to_csv(RESULTADOS / 'robustez_RF_anidada.csv', index=False)

tabla = (por_cfg.groupby(['max_depth', 'min_samples_leaf']).auc_roc
         .agg(['mean', 'std']).round(3).reset_index())
base = por_cfg[(por_cfg.max_depth == 'None') & (por_cfg.min_samples_leaf == 1)].auc_roc

with open(RESULTADOS / 'robustez_RF_resumen.txt', 'w', encoding='utf-8') as f:
    def out(s=''):
        print(s)
        f.write(s + '\n')
    out('1) Matriz de confusión por paciente, repetición 0 (616 pacientes, cada una una vez)')
    out(f'   VN={tn}  FP={fp}  FN={fn}  VP={tp}')
    out(f'   precisión={tp / (tp + fp):.3f}  recall={tp / (tp + fn):.3f}  '
        f'especificidad={tn / (tn + fp):.3f}  exactitud={(tp + tn) / cm.sum():.3f}')
    out()
    out('2a) AUC-ROC por paciente de cada configuración (25 pliegues, media y desv. est.)')
    out(tabla.to_string(index=False))
    out(f'   Rango de medias: {tabla["mean"].min():.3f} a {tabla["mean"].max():.3f}')
    out()
    out('2b) Validación anidada (configuración elegida dentro del entrenamiento)')
    out(f'   AUC-ROC externo: {anidada.auc_roc.mean():.3f} ± {anidada.auc_roc.std():.3f}')
    out(f'   Modelo de la Tabla 4.1 (max_depth=None, min_samples_leaf=1): '
        f'{base.mean():.3f} ± {base.std():.3f}')
    out('   Configuraciones elegidas:')
    out(anidada.groupby(['max_depth', 'min_samples_leaf']).size()
        .rename('veces').reset_index().to_string(index=False))